# 31 — Dim Sweep: 256 / 512 / 1024

Runs TripletAE with out_dim ∈ [256, 512, 1024] sequentially.
512 checkpoint already exists — training is skipped for it automatically.

In [ ]:
import os, random, sys, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
sys.path.append('/raid/ruban/hpmlproj/term_project/SigSpatial')
from sota_experiment_common import (
    build_fn_mask, build_gt_cache, build_gt_gpu,
    cleanup, eval_recall, l1_simplex, load_dataset,
    nmslib_neighbors, preload_rerank_corpus, release_rerank_corpus, rerank_wj_gpu, save_result,
)

# ── sweep config ─────────────────────────────────────────────────────────────
DIMS         = [256, 512, 1024]   # will skip training if checkpoint already exists
dataset_name = "full"
batch_size   = 2048
epochs       = 50
lr           = 1e-3
weight_decay = 1e-4
max_pos      = 30
margin       = 0.3
lambda_recon = 0.1
THREADS      = 150
seed         = 42
candidate_ks = [1000, 2000]
NOTEBOOK_NAME = "31_dim_sweep.ipynb"

random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
print(f"GPUs={torch.cuda.device_count()}  dims={DIMS}")

In [ ]:
qt, gt, query_start, corpus_qt, query_qt, corpus_sums = load_dataset(dataset_name)
qt_norm = l1_simplex(qt.copy())

device      = torch.device("cuda:0")
vecs_device = torch.device("cuda:7")
print("Pre-loading vectors to cuda:7...")
vecs_gpu = torch.from_numpy(np.ascontiguousarray(qt_norm, dtype=np.float32)).to(vecs_device)
print(f"vecs_gpu: {vecs_gpu.nbytes/1024**3:.2f} GB on {vecs_device}")

gt_stacked = build_gt_cache(gt, len(qt_norm), query_start, dataset_name)
gt_gpu     = build_gt_gpu(gt_stacked, vecs_device)
del gt_stacked

In [ ]:
def wj_sim(a, b):
    mins = torch.minimum(a, b).sum(dim=-1)
    maxs = torch.maximum(a, b).sum(dim=-1).clamp(min=1e-10)
    return mins / maxs

def wj_triplet_loss_inbatch(anchors, positives, margin=0.3, gt_matrix=None):
    sim_ap = wj_sim(anchors, positives)
    mins_c = torch.min(anchors.unsqueeze(1), positives.unsqueeze(0)).sum(2)
    maxs_c = torch.max(anchors.unsqueeze(1), positives.unsqueeze(0)).sum(2)
    sim_cross = mins_c / maxs_c.clamp(min=1e-10)
    sim_cross.fill_diagonal_(-1e9)
    n_fn = 0
    if gt_matrix is not None:
        fn_mask = gt_matrix.to(sim_cross.device)
        n_fn = int(fn_mask.sum().item())
        if n_fn:
            sim_cross[fn_mask] = -1e9
    sim_an   = sim_cross.max(dim=1).values
    loss     = F.relu(sim_an - sim_ap + margin)
    violated = loss > 0
    if violated.sum() == 0:
        return torch.tensor(0.0, device=anchors.device, requires_grad=True), 0, n_fn
    return loss[violated].mean(), int(violated.sum().item()), n_fn

class IndexAnchorPositiveDataset(Dataset):
    def __init__(self, gt_lookup, query_start, max_pos=30):
        self.pairs = []
        for qid, neighbors in gt_lookup.items():
            for nid in neighbors[:max_pos]:
                if qid >= query_start and nid < query_start:
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"pairs={len(self.pairs):,}  steps/epoch={len(self.pairs)//batch_size}")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        qid, pid = self.pairs[idx]
        return qid, pid

class TripletAE(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(out_dim, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, in_dim, bias=False),
        )
    def encode(self, x):
        z = F.relu(self.encoder(x))
        return z / z.sum(dim=1, keepdim=True).clamp(min=1e-10)
    def forward(self, x):
        z = self.encode(x)
        return z, F.relu(self.decoder(z))

def embed_all(model, qt, device, batch_size=512):
    enc = model.module if hasattr(model, 'module') else model
    enc.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(qt), batch_size):
            x = torch.tensor(qt[s:s+batch_size], dtype=torch.float32, device=device)
            out.append(enc.encode(x).cpu().numpy().astype(np.float32))
    return np.vstack(out)

def eval_dim(model, out_dim, device):
    out_path    = f"/tmp/results_sota_triplet_autoencoder_wj_{out_dim}.pkl"
    method_name = f"triplet_autoencoder_wj_{out_dim}"

    embs        = embed_all(model, qt_norm, device)
    corpus_embs = embs[:query_start]
    query_embs  = embs[query_start:]

    # ── no-rerank ────────────────────────────────────────────────────────────
    max_k = max(max(candidate_ks), 500)
    nbrs, info = nmslib_neighbors(corpus_embs, query_embs,
                                   space="WeightedJaccard", k=max_k, threads=THREADS)
    metrics = {**eval_recall(gt, nbrs, query_start, max_k), **info, "dim": out_dim}
    print(f"\n[dim={out_dim}] no-rerank")
    for k, v in metrics.items():
        if isinstance(k, int): print(f"  R@{k:<4} = {v:.4f}")
    print(f"  QPS={metrics['qps']:.1f}")
    save_result(out_path, dataset_name, method_name, metrics, meta={"notebook": NOTEBOOK_NAME})

    # ── rerank ───────────────────────────────────────────────────────────────
    preload_rerank_corpus(corpus_qt, corpus_sums)
    for ck in candidate_ks:
        cand, ci = nmslib_neighbors(corpus_embs, query_embs,
                                     space="WeightedJaccard", k=ck, threads=THREADS)
        t0  = time.time()
        rr  = rerank_wj_gpu(query_qt, cand, corpus_qt, corpus_sums, top_k=ck, batch_size=8)
        qps = len(query_qt) / max(time.time()-t0 + len(query_qt)/max(ci['qps'],1e-9), 1e-9)
        rr_metrics = {**eval_recall(gt, rr, query_start, ck), "qps": qps, "candidate_k": ck}
        key = f"{method_name}_rerank_{ck}"
        print(f"\n[dim={out_dim}] rerank@{ck}")
        for k, v in rr_metrics.items():
            if isinstance(k, int): print(f"  {key} R@{k} = {v:.4f}")
        print(f"  {key} QPS={rr_metrics['qps']:.1f}")
        save_result(out_path, dataset_name, key, rr_metrics, meta={"notebook": NOTEBOOK_NAME})
    release_rerank_corpus()

    # ── 10k eval ─────────────────────────────────────────────────────────────
    qt_10k, gt_10k, qs_10k, corpus_qt_10k, query_qt_10k, corpus_sums_10k = load_dataset("10k")
    qt_10k_norm = l1_simplex(qt_10k.copy())
    embs_10k    = embed_all(model, qt_10k_norm, device)
    corpus_embs_10k = embs_10k[:qs_10k]
    query_embs_10k  = embs_10k[qs_10k:]
    cand_ks_10k = [500, 1000]
    max_k_10k   = max(max(cand_ks_10k), 500)
    nbrs_10k, info_10k = nmslib_neighbors(corpus_embs_10k, query_embs_10k,
                                            space="WeightedJaccard", k=max_k_10k, threads=THREADS)
    m10k = {**eval_recall(gt_10k, nbrs_10k, qs_10k, max_k_10k), **info_10k, "dim": out_dim}
    print(f"\n[dim={out_dim}] 10k no-rerank")
    for k, v in m10k.items():
        if isinstance(k, int): print(f"  R@{k:<4} = {v:.4f}")
    print(f"  QPS={m10k['qps']:.1f}")
    save_result(out_path, "10k", method_name, m10k, meta={"notebook": NOTEBOOK_NAME})

    preload_rerank_corpus(corpus_qt_10k, corpus_sums_10k)
    for ck in cand_ks_10k:
        cand_10k, ci_10k = nmslib_neighbors(corpus_embs_10k, query_embs_10k,
                                              space="WeightedJaccard", k=ck, threads=THREADS)
        t0   = time.time()
        rr10 = rerank_wj_gpu(query_qt_10k, cand_10k, corpus_qt_10k, corpus_sums_10k, top_k=ck, batch_size=64)
        qps10 = len(query_qt_10k) / max(time.time()-t0 + len(query_qt_10k)/max(ci_10k['qps'],1e-9), 1e-9)
        rr10_metrics = {**eval_recall(gt_10k, rr10, qs_10k, ck), "qps": qps10, "candidate_k": ck}
        key10 = f"{method_name}_rerank_{ck}"
        print(f"\n[dim={out_dim}] 10k rerank@{ck}")
        for k, v in rr10_metrics.items():
            if isinstance(k, int): print(f"  {key10} R@{k} = {v:.4f}")
        print(f"  {key10} QPS={rr10_metrics['qps']:.1f}")
        save_result(out_path, "10k", key10, rr10_metrics, meta={"notebook": NOTEBOOK_NAME})
    release_rerank_corpus()

print("Definitions ready.")

In [ ]:
dataset_obj = IndexAnchorPositiveDataset(gt, query_start, max_pos=max_pos)

for out_dim in DIMS:
    ckpt = f"/tmp/best_sota_triplet_autoencoder_wj_{out_dim}_full.pt"
    print(f"\n{'='*60}")
    print(f"dim={out_dim}  ckpt={'EXISTS — skipping training' if Path(ckpt).exists() else 'MISSING — will train'}")
    print('='*60)

    model = TripletAE(qt_norm.shape[1], out_dim)
    model = nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))
    model = model.to(device)

    if not Path(ckpt).exists():
        loader = DataLoader(dataset_obj, batch_size=batch_size, shuffle=True,
                            num_workers=4, pin_memory=True, drop_last=True,
                            persistent_workers=True)
        opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        best = float('inf')
        t0_train = time.time()
        for epoch in range(1, epochs + 1):
            model.train()
            tot_loss = tot_trip = tot_rec = tot_viol = steps = 0
            for a_ids, p_ids in tqdm(loader, desc=f"dim={out_dim} ep{epoch:02d}", leave=False):
                a = vecs_gpu[a_ids.to(vecs_device)].to(device)
                p = vecs_gpu[p_ids.to(vecs_device)].to(device)
                B = a.shape[0]
                out = model(torch.cat([a, p]))
                za, zp   = out[0][:B], out[0][B:]
                rec_a, rec_p = out[1][:B], out[1][B:]
                fn_mask  = build_fn_mask(a_ids, p_ids, gt_gpu, query_start)
                trip, n_viol, _ = wj_triplet_loss_inbatch(za, zp, margin=margin, gt_matrix=fn_mask)
                rec_loss = (F.mse_loss(rec_a, a) + F.mse_loss(rec_p, p)) * 0.5
                loss     = trip + lambda_recon * rec_loss
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
                tot_loss += float(loss.detach()); tot_trip += float(trip.detach())
                tot_rec  += float(rec_loss.detach()); tot_viol += n_viol; steps += 1
            sch.step()
            avg = tot_loss / max(steps, 1)
            if avg < best:
                best = avg
                torch.save(model.module.state_dict(), ckpt)
            elapsed = (time.time() - t0_train) / 60
            eta     = elapsed / epoch * (epochs - epoch)
            if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
                print(f"  dim={out_dim} ep{epoch:02d}/{epochs} loss={avg:.4f} trip={tot_trip/steps:.4f} "
                      f"rec={tot_rec/steps:.6f} viol={tot_viol/steps:.1f} "
                      f"{elapsed:.1f}min eta={eta:.1f}min", flush=True)
        print(f"  Training done. best={best:.4f} saved {ckpt}")

    # load best checkpoint and eval
    (model.module if hasattr(model, 'module') else model).load_state_dict(
        torch.load(ckpt, map_location=device, weights_only=True))
    eval_dim(model, out_dim, device)
    del model; torch.cuda.empty_cache()

print("\nAll dims done.")
cleanup()